In [ ]:
# Interactive Plotly 3D scatter — one or more partitions; xyz from POINTS_XYZ_NPY + global_node_ids.

from pathlib import Path
import numpy as np

try:
    import plotly.graph_objects as go
except ImportError as e:
    raise RuntimeError("Install plotly: conda/pip install plotly") from e

POINTS_XYZ_NPY = Path(
    "/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_points_xyz.npy"
)

NODE_SUBSAMPLE = 30_000

test = np.load(POINTS_XYZ_NPY)

# generate random subsample of points
subsample_idx = np.random.randint(0, test.shape[0], size=NODE_SUBSAMPLE)
subsample = test[subsample_idx]

# Plot subsample of points as scatter on interactive 3D plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=subsample[:, 0],
            y=subsample[:, 1],
            z=subsample[:, 2],
            mode='markers',
            marker=dict(
                size=2,
                color='blue',
                opacity=0.1,
                symbol='circle',
            ),
        )
    ]
)

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    title="3D Point Cloud Subsample (Interactive)"
)
fig.show()


In [ ]:
# Plot one or more partition NPZs in 3D (Plotly).
#
# `x` in the NPZ is *not* xyz — use POINTS_XYZ_NPY + global_node_ids.

import colorsys
from pathlib import Path

import numpy as np

try:
    import plotly.graph_objects as go
except Exception as e:
    raise RuntimeError("Install plotly: conda/pip install plotly") from e

# ---------------- USER INPUT ----------------
# One partition: length-1 list. Multiple: add more paths (same graph / same points file).
PARTITION_NPZS = [
    Path(
        "/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/"
        "partition_test_000000.npz"
    ),

    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000011.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000012.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000013.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000014.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000015.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000016.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000017.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000018.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000019.npz"),
    Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000020.npz"),

    
    
    
    
]
# Or build from a directory, e.g.:
# PARTITION_NPZS = sorted(Path(".../partitions_dir").glob("partition_*.npz"))[:8]

POINTS_XYZ_NPY = Path(
    "/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_points_xyz.npy"
)

NODE_SUBSAMPLE = 10_000   # per partition (local nodes)
EDGE_SUBSAMPLE = 10_000  # per partition (after endpoint filter)
RNG_SEED = 42
DRAW_EDGES = True

WRITE_HTML = False
HTML_OUT = Path("/pscratch/sd/d/dkololgi/abacus/partitions/partition_multi_preview.html")
# -------------------------------------------

def _part_colors(n: int):
    hues = (np.arange(n) * 0.618033988749895) % 1.0
    cols = []
    for i in range(n):
        h = float(hues[i])
        r, g, b = colorsys.hsv_to_rgb(h, 0.75, 0.95)
        cols.append(f"rgb({int(255*r)},{int(255*g)},{int(255*b)})")
    return cols


pts_path = POINTS_XYZ_NPY.expanduser().resolve()
if not pts_path.exists():
    raise FileNotFoundError(f"Missing points: {pts_path}")
pts_all = np.load(pts_path).astype(np.float64)
if pts_all.ndim != 2 or pts_all.shape[1] < 3:
    raise ValueError(f"points_xyz must be (N,>=3); got {pts_all.shape}")

paths = [Path(x).expanduser().resolve() for x in PARTITION_NPZS]
for p in paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

rng = np.random.default_rng(int(RNG_SEED))
colors = _part_colors(len(paths))

fig = go.Figure()
titles = []

for pi, p in enumerate(paths):
    c_edge = colors[pi]
    with np.load(p, allow_pickle=False) as d:
        global_node_ids = np.asarray(d["global_node_ids"], dtype=np.int64)
        core_mask_local = np.asarray(d["core_mask_local"], dtype=bool)
        edge_index = np.asarray(d["edge_index"], dtype=np.int64)

    n = int(global_node_ids.size)
    gmax = int(global_node_ids.max()) if n else -1
    if gmax >= pts_all.shape[0]:
        raise ValueError(f"{p.name}: global_node_ids max={gmax} vs points rows={pts_all.shape[0]}")

    xyz_full = pts_all[global_node_ids, :3]
    if not np.isfinite(xyz_full).all():
        raise ValueError(f"{p.name}: non-finite xyz")

    if NODE_SUBSAMPLE is None or int(NODE_SUBSAMPLE) >= n:
        keep_local = np.arange(n, dtype=np.int64)
    else:
        keep_local = np.sort(rng.choice(n, size=int(NODE_SUBSAMPLE), replace=False))

    vis = np.zeros(n, dtype=bool)
    vis[keep_local] = True
    xyz = xyz_full[keep_local]
    core_sub = core_mask_local[keep_local]
    inv = np.full(n, -1, dtype=np.int32)
    inv[keep_local] = np.arange(keep_local.size, dtype=np.int32)

    titles.append(f"{p.name}: nodes {keep_local.size:,}/{n:,}")

    if DRAW_EDGES and edge_index.size > 0:
        s, r = edge_index[0], edge_index[1]
        m = vis[s] & vis[r]
        s_f, r_f = s[m], r[m]
        E = int(s_f.size)
        if EDGE_SUBSAMPLE is not None and int(EDGE_SUBSAMPLE) > 0 and E > int(EDGE_SUBSAMPLE):
            sel = rng.choice(E, size=int(EDGE_SUBSAMPLE), replace=False)
            s_f, r_f = s_f[sel], r_f[sel]
            E = int(s_f.size)
        si, ri = inv[s_f], inv[r_f]
        xl = np.empty(3 * E, dtype=np.float64)
        yl = np.empty(3 * E, dtype=np.float64)
        zl = np.empty(3 * E, dtype=np.float64)
        xl[::3], yl[::3], zl[::3] = xyz[si, 0], xyz[si, 1], xyz[si, 2]
        xl[1::3], yl[1::3], zl[1::3] = xyz[ri, 0], xyz[ri, 1], xyz[ri, 2]
        xl[2::3] = yl[2::3] = zl[2::3] = np.nan
        fig.add_trace(
            go.Scatter3d(
                x=xl, y=yl, z=zl, mode="lines",
                line=dict(color=c_edge, width=2), opacity=0.18, hoverinfo="skip",
                name=f"[{pi}] edges {p.stem} ({E:,})",
            )
        )

    halo_idx = np.flatnonzero(~core_sub)
    core_idx = np.flatnonzero(core_sub)
    if halo_idx.size:
        fig.add_trace(
            go.Scatter3d(
                x=xyz[halo_idx, 0], y=xyz[halo_idx, 1], z=xyz[halo_idx, 2],
                mode="markers", marker=dict(size=2, color=c_edge, opacity=0.35),
                hoverinfo="skip", name=f"[{pi}] halo {p.stem} ({halo_idx.size:,})",
            )
        )
    fig.add_trace(
        go.Scatter3d(
            x=xyz[core_idx, 0], y=xyz[core_idx, 1], z=xyz[core_idx, 2],
            mode="markers", marker=dict(size=4, color=c_edge, opacity=0.85),
            hoverinfo="skip", name=f"[{pi}] core {p.stem} ({core_idx.size:,})",
        )
    )

# Scene limits from all traces would need collecting xyz — recompute bbox from files cheaply:
all_min = np.array([np.inf, np.inf, np.inf])
all_max = np.array([-np.inf, -np.inf, -np.inf])
for p in paths:
    with np.load(p, allow_pickle=False) as d:
        g = np.asarray(d["global_node_ids"], dtype=np.int64)
    xyzp = pts_all[g, :3]
    all_min = np.minimum(all_min, xyzp.min(axis=0))
    all_max = np.maximum(all_max, xyzp.max(axis=0))
pad = 0.03 * max(float(np.max(all_max - all_min)), 1e-6)
lo, hi = (all_min - pad).tolist(), (all_max + pad).tolist()

fig.update_layout(
    title="<br>".join(["METIS partition(s) — comoving xyz"] + titles),
    scene=dict(
        xaxis=dict(title="x [Mpc]", backgroundcolor="#0a0a0c", showgrid=False, zeroline=False),
        yaxis=dict(title="y [Mpc]", backgroundcolor="#0a0a0c", showgrid=False, zeroline=False),
        zaxis=dict(title="z [Mpc]", backgroundcolor="#0a0a0c", showgrid=False, zeroline=False),
        aspectmode="cube",
    ),
    paper_bgcolor="#0a0a0c", font=dict(color="white"),
    margin=dict(l=0, r=0, t=80, b=0), legend=dict(x=0.02, y=0.98),
    width=1000, height=820,
)

if WRITE_HTML:
    HTML_OUT.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(HTML_OUT), include_plotlyjs="cdn")
    print(f"Wrote HTML: {HTML_OUT}")

try:
    fig.show(renderer="notebook_connected")
except Exception:
    fig.show()

In [ ]:
test2.files

In [ ]:
part = Path("/pscratch/sd/d/dkololgi/abacus/partitions/abacus_23032026_transformed_eig_halo4_core50k/partition_train_000031.npz")

test2 = np.load(part)

test2['global_node_ids']
partition = test[test2['global_node_ids']]
# plot the partition
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=partition[:, 0], y=partition[:, 1], z=partition[:, 2],
    mode='markers',
    marker=dict(size=2, color='blue', opacity=0.5)
))
fig.update_layout(scene=dict(
    xaxis=dict(range=[-2000, 2000]),
    yaxis=dict(range=[-2000, 2000]),
    zaxis=dict(range=[-2000, 2000])
))
fig.show()